# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll work with a Croissant schema describing mass spectrometry protein data.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets and their fields using their @id
print("Available Record Sets and Fields:")
for rset in dataset.record_sets():
    print(f"\nRecord Set: {rset.name} (id={rset.id})")
    print("Fields (id | name | dataType):")
    for field in rset.fields:
        dtype = getattr(field, 'data_type', 'N/A')
        print(f"- {field.id} | {field.name} | {dtype}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get all record set @id's (as a list)
record_sets = [rset.id for rset in dataset.record_sets()]
print("Record Sets IDs:", record_sets)

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")

# For illustration, pick the first record set for data exploration
main_record_set_id = record_sets[0]
print(f"\nColumn list for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify numeric fields (based on dataType or manual inspection)
main_df = dataframes[main_record_set_id]
print("Numeric columns detected (dtype):")
numeric_cols = main_df.select_dtypes(include='number').columns.tolist()
print(numeric_cols)

# For demonstration, we'll pick the first numeric field for numeric analysis
if numeric_cols:
    numeric_field_id = numeric_cols[0]
    threshold = main_df[numeric_field_id].mean()  # Use mean as demo threshold

    # Filter on the numeric field
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize this field (z-score)
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If there are categorical fields, attempt to group
    category_cols = main_df.select_dtypes(include='object').columns.tolist()
    if category_cols:
        group_field_id = category_cols[0]  # Choose the first categorical field
        print(f"\nGrouping by {group_field_id} and summarizing {numeric_field_id}:")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(grouped_df.head())
else:
    print("No numeric fields detected in the main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For demonstration, plot the distribution of the first detected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of numeric field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a group_field exists, plot a boxplot
    if category_cols:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook loaded the FAIR² mass spectrometry protein dataset using `mlcroissant`, examined available record sets and their fields, loaded records to Pandas DataFrames, and performed basic exploratory analyses. For more advanced tasks, further domain expertise and data review are recommended.